In [1]:
!pip install rank_bm25 sentence-transformers scikit-learn networkx pymupdf python-docx -q

import os
import re
import json
import random
import numpy as np
import networkx as nx
import pandas as pd
from pathlib import Path
from collections import defaultdict
from rank_bm25 import BM25Okapi
from sklearn.covariance import LedoitWolf
from sklearn.metrics.pairwise import euclidean_distances, cosine_distances
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, pipeline
import torch
import fitz
from docx import Document as DocxDocument

random.seed(42)
print(f"GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 67.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 15.2 MB/s eta 0:00:00
GPU: Tesla T4


CELL 2 — Load BGE

In [2]:
print("Loading BGE...")
model     = SentenceTransformer("BAAI/bge-base-en-v1.5", device="cuda")
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5")
print("Done.")

Loading BGE...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Done.


CELL 3 — All Functions

In [3]:
def embed_query(text, model, tokenizer, chunk_size=400, overlap=50):
    tokens = tokenizer(
        text, truncation=False, return_tensors=None
    )["input_ids"]
    if len(tokens) <= 512:
        return model.encode(
            [f"Represent this sentence for searching relevant passages: {text}"],
            normalize_embeddings=True, convert_to_numpy=True
        )[0]
    chunks, start = [], 0
    while start < len(tokens):
        end   = min(start + chunk_size, len(tokens))
        chunk = tokenizer.decode(tokens[start:end], skip_special_tokens=True)
        chunks.append(
            f"Represent this sentence for searching relevant passages: {chunk}"
        )
        if end == len(tokens): break
        start += chunk_size - overlap
    embs = model.encode(chunks, normalize_embeddings=True, convert_to_numpy=True)
    mean = embs.mean(axis=0)
    return mean / np.linalg.norm(mean)

def semantic_retrieve(q_emb, corpus_embeddings, top_k=10):
    scores = corpus_embeddings @ q_emb
    ranked = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in ranked]

def evaluate(retrieved_dict, relevance, query_ids,
             top_k_list=[5,10], label=""):
    ap_scores = []
    recalls   = {k: [] for k in top_k_list}
    for q_id in query_ids:
        if q_id not in relevance or not relevance[q_id]: continue
        relevant = relevance[q_id]
        ranked   = [r[0] for r in retrieved_dict[q_id]]
        hits, ap = 0, 0.0
        for rank, idx in enumerate(ranked, 1):
            if idx in relevant:
                hits += 1
                ap   += hits / rank
        ap_scores.append(ap / len(relevant))
        for k in top_k_list:
            recalls[k].append(len(set(ranked[:k]) & relevant) / len(relevant))
    results = {
        "MAP"      : round(np.mean(ap_scores), 4),
        "Recall@5" : round(np.mean(recalls[5]), 4),
        "Recall@10": round(np.mean(recalls[10]), 4),
    }
    if label:
        print(f"\n=== {label} ===")
        for k, v in results.items():
            print(f"  {k:12}: {v}")
    return results

class LegalDocumentGraph:
    def __init__(self, embeddings, metric="mahalanobis"):
        self.embeddings = embeddings
        self.N          = len(embeddings)
        self.metric     = metric
        self.k          = max(3, int(np.ceil(np.log(self.N))))
        print(f"Building {metric} graph | N={self.N} | K={self.k}")
        self.dist_matrix = self._compute_distances()
        self.graph       = self._build_graph()
        print(f"Edges: {self.graph.number_of_edges()}")

    def _compute_distances(self):
        if self.metric == "euclidean":
            return euclidean_distances(self.embeddings)
        elif self.metric == "cosine":
            return cosine_distances(self.embeddings)
        elif self.metric == "mahalanobis":
            print("  LedoitWolf...")
            lw = LedoitWolf()
            lw.fit(self.embeddings)
            L  = np.linalg.cholesky(lw.get_precision())
            return euclidean_distances(self.embeddings @ L)

    def _build_graph(self):
        G = nx.Graph()
        G.add_nodes_from(range(self.N))
        for i in range(self.N):
            for j in np.argsort(self.dist_matrix[i])[1:self.k+1]:
                G.add_edge(i, j, weight=1.0/(self.dist_matrix[i][j]+1e-9))
        return G

    def get_neighbors(self, node, max_hops=2):
        return {n: d for n, d in
                nx.single_source_shortest_path_length(
                    self.graph, node, cutoff=max_hops
                ).items() if n != node}

    def get_edge_weight(self, i, j):
        return self.graph[i][j]["weight"] if self.graph.has_edge(i,j) else 0.0

class SSHRetriever:
    def __init__(self, corpus_texts, corpus_embeddings, graph,
                 query_model, query_tokenizer,
                 decay_lambda=0.7, alpha=0.5,
                 bm25_top_k=20, sem_top_k=20, max_hops=2, top_k=10):
        self.corpus_texts      = corpus_texts
        self.corpus_embeddings = corpus_embeddings
        self.graph             = graph
        self.query_model       = query_model
        self.query_tokenizer   = query_tokenizer
        self.decay_lambda      = decay_lambda
        self.alpha             = alpha
        self.bm25_top_k        = bm25_top_k
        self.sem_top_k         = sem_top_k
        self.max_hops          = max_hops
        self.top_k             = top_k
        self.bm25 = BM25Okapi([t.lower().split() for t in corpus_texts])

    def _bm25_anchors(self, query_text):
        scores = self.bm25.get_scores(query_text.lower().split())
        ranked = np.argsort(scores)[::-1][:self.bm25_top_k]
        s      = scores[ranked]
        if s.max() > 0: s = s / s.max()
        return ranked, s

    def _semantic_anchors(self, q_emb):
        scores = self.corpus_embeddings @ q_emb
        ranked = np.argsort(scores)[::-1][:self.sem_top_k]
        s      = scores[ranked]
        if s.max() > s.min():
            s = (s - s.min()) / (s.max() - s.min())
        return ranked, s

    def _propagate_query_aware(self, anchor_indices, anchor_scores, q_emb):
        graph_scores = defaultdict(float)
        for idx, score in zip(anchor_indices, anchor_scores):
            graph_scores[int(idx)] += float(score)
            for neighbor, hops in self.graph.get_neighbors(
                int(idx), self.max_hops
            ).items():
                static_w     = self.graph.get_edge_weight(int(idx), neighbor)
                q_rel_a      = float(self.corpus_embeddings[int(idx)] @ q_emb)
                q_rel_n      = float(self.corpus_embeddings[neighbor] @ q_emb)
                query_boost  = max(0, (q_rel_a + q_rel_n) / 2)
                combined_w   = static_w * (1 + query_boost)
                graph_scores[neighbor] += (
                    float(score) * combined_w * (self.decay_lambda ** hops)
                )
        return graph_scores

    def retrieve_from_embedding(self, q_emb, query_text):
        bm25_idx, bm25_sc = self._bm25_anchors(query_text)
        sem_idx,  sem_sc  = self._semantic_anchors(q_emb)
        bm25_g = self._propagate_query_aware(bm25_idx, bm25_sc, q_emb)
        sem_g  = self._propagate_query_aware(sem_idx,  sem_sc,  q_emb)
        merged = {
            c: (bm25_g.get(c,0.0) + sem_g.get(c,0.0)) / 2.0
            for c in set(bm25_g) | set(sem_g)
        }
        cand_list = list(merged.keys())
        cand_embs = self.corpus_embeddings[cand_list]
        sem_final = cand_embs @ q_emb
        g = np.array([merged[c] for c in cand_list])
        if g.max() > 0: g = g / g.max()
        s = sem_final
        if s.max() > s.min():
            s = (s - s.min()) / (s.max() - s.min())
        final   = self.alpha * s + (1 - self.alpha) * g
        top_idx = np.argsort(final)[::-1][:self.top_k]
        return [(cand_list[i], float(final[i])) for i in top_idx]

print("All functions defined.")

All functions defined.


CELL 4 — Load Documents and Chunk

In [4]:
def load_document(filepath):
    ext = Path(filepath).suffix.lower()
    if ext == ".pdf":
        doc  = fitz.open(filepath)
        text = "".join(page.get_text() for page in doc)
        doc.close()
        return text
    elif ext == ".docx":
        doc = DocxDocument(filepath)
        return "\n".join(p.text for p in doc.paragraphs if p.text.strip())
    elif ext == ".txt":
        for enc in ["utf-8", "latin-1", "cp1252"]:
            try:
                with open(filepath, "r", encoding=enc, errors="ignore") as f:
                    return f.read()
            except: continue
    return ""

def chunk_text(text, doc_name, chunk_size=256, overlap=50):
    words  = text.split()
    chunks = []
    start, idx = 0, 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append({
            "chunk_id": f"{doc_name}_chunk_{idx}",
            "doc_name": doc_name,
            "text"    : " ".join(words[start:end])
        })
        if end == len(words): break
        start += chunk_size - overlap
        idx   += 1
    return chunks

def load_all_documents(doc_dir):
    all_chunks = []
    files = [f for f in os.listdir(doc_dir)
             if f.endswith((".pdf", ".docx", ".txt"))]
    print(f"Found {len(files)} documents")
    for fname in files:
        text   = load_document(os.path.join(doc_dir, fname))
        if not text.strip(): continue
        chunks = chunk_text(text, Path(fname).stem)
        all_chunks.extend(chunks)
        print(f"  {len(chunks):4d} chunks — {fname}")
    print(f"Total: {len(all_chunks)} chunks")
    return all_chunks

# Update path to your documents
doc_dir    = "/kaggle/input/datasets/sauraviitjapdcl/duomind-sshrag-mahalanobis"
all_chunks = load_all_documents(doc_dir)
chunk_texts= [c["text"] for c in all_chunks]
chunk_ids  = [c["chunk_id"] for c in all_chunks]

Found 20 documents
    40 chunks — 24062022_Guidelines_on_Cyber_Security_in_Power_Sector_2021.pdf
    72 chunks — NCIIPC_Guidelines_V2.pdf
     9 chunks — CERT-In_Directions.docx
    52 chunks — DPDP ACT.pdf
    13 chunks — State Data Centre security guidelines.pdf
    50 chunks — DPDP_Rules_2025_English_only.pdf
   271 chunks — CELEX_32016R0679_EN_TXT.pdf
    10 chunks — Presentation - CSCRF.pdf
   213 chunks — e-Governance Policy Initiatives.pdf
   200 chunks — Electricity Act 2003.pdf
    74 chunks — guidelinesgovtentities.pdf
    33 chunks — Cyber Security Framework in Banks.docx
    35 chunks — ISO_IEC-270012022-ed.3.pdf
    38 chunks — nist cyber security framework.docx
    20 chunks — trai_takshalia.pdf
   205 chunks — NIST.SP.800-66r2.pdf
    20 chunks — OECD-LEGAL-0479-en.docx
     6 chunks — pcissc_overview.pdf
    20 chunks — NCIIPC-Rules-notification-1.pdf
    95 chunks — IT ACT 2000.pdf
Total: 1476 chunks


CELL 5 — Load QA and Build Gold Labels

In [5]:
# Load 340Q
qa_df = pd.read_csv(
    "/kaggle/input/datasets/sauraviitjapdcl/ssh-rag-code/ssh_rag_groundtruth.csv",
    encoding="cp1252"
)
qa_df_clean = qa_df.drop_duplicates(subset='question', keep='first')
print(f"Questions: {len(qa_df_clean)}")

# Embed corpus
print("Embedding corpus...")
duomind_embeddings = model.encode(
    chunk_texts, batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

# Embed answers
answer_texts = qa_df_clean["ground_truth_answer"].tolist()
print("Embedding answers...")
answer_embeddings = model.encode(
    answer_texts, batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

# Build gold labels
gold_labels = {}
for i in range(len(qa_df_clean)):
    question = qa_df_clean.iloc[i]["question"]
    source   = str(qa_df_clean.iloc[i]["document_source"])
    scores   = duomind_embeddings @ answer_embeddings[i]
    is_cross = '+' in source or 'Cross' in source
    if is_cross:
        gold_labels[question] = set(np.argsort(scores)[::-1][:2].tolist())
    else:
        gold_labels[question] = {int(np.argmax(scores))}

print(f"Gold labels: {len(gold_labels)}")

# Split
all_qids    = list(gold_labels.keys())
random.shuffle(all_qids)
dm_test_ids = all_qids[int(len(all_qids)*0.8):]
dm_test_r   = {q: gold_labels[q] for q in dm_test_ids}
print(f"Test questions: {len(dm_test_ids)}")

# Ground truth lookup
gt_lookup = dict(zip(
    qa_df_clean["question"].tolist(),
    qa_df_clean["ground_truth_answer"].tolist()
))

Questions: 378
Embedding corpus...


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Embedding answers...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Gold labels: 378
Test questions: 76


Cell 6 — Load Both Models and Build Both Graphs

In [6]:
# Original BGE
print("Original BGE already loaded as 'model'")

# Load fine-tuned BGE
print("Loading fine-tuned BGE...")
from sentence_transformers import SentenceTransformer
ft_bge = SentenceTransformer(
    "/kaggle/input/models/sauraviitjapdcl/finetuned-bge/other/default/1",
    device="cuda"
)
print("Fine-tuned BGE loaded.")

# Embed test queries — original BGE
print("\nEmbedding test queries (original BGE)...")
dm_q_embs_orig = {}
for q in dm_test_ids:
    dm_q_embs_orig[q] = embed_query(q, model, tokenizer)
print(f"Done. {len(dm_q_embs_orig)} queries.")

# Embed test queries — fine-tuned BGE
print("\nEmbedding test queries (fine-tuned BGE)...")
dm_q_embs_ft = {}
for q in dm_test_ids:
    dm_q_embs_ft[q] = ft_bge.encode(
        [f"Represent this sentence for searching relevant passages: {q}"],
        normalize_embeddings=True,
        convert_to_numpy=True
    )[0]
print(f"Done. {len(dm_q_embs_ft)} queries.")

# Re-embed corpus with fine-tuned BGE
print("\nRe-embedding corpus with fine-tuned BGE...")
duomind_emb_ft = ft_bge.encode(
    chunk_texts,
    batch_size           = 64,
    show_progress_bar    = True,
    normalize_embeddings = True,
    convert_to_numpy     = True
)
print(f"Shape: {duomind_emb_ft.shape}")

# Build graphs — both original and fine-tuned
print("\nBuilding Mahalanobis graph (original BGE)...")
graph_maha_orig = LegalDocumentGraph(duomind_embeddings, "mahalanobis")

print("\nBuilding Mahalanobis graph (fine-tuned BGE)...")
graph_maha_ft = LegalDocumentGraph(duomind_emb_ft, "mahalanobis")

# Build retrievers — all four variants
print("\nBuilding retrievers...")

# 1. SSH-RAG original BGE
ret_ssh_orig = SSHRetriever(
    chunk_texts, duomind_embeddings, graph_maha_orig,
    model, tokenizer, decay_lambda=0.7, alpha=0.5
)

# 2. SSH-RAG fine-tuned BGE
ret_ssh_ft = SSHRetriever(
    chunk_texts, duomind_emb_ft, graph_maha_ft,
    ft_bge, tokenizer, decay_lambda=0.7, alpha=0.5
)

print("All retrievers ready.")

Original BGE already loaded as 'model'
Loading fine-tuned BGE...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fine-tuned BGE loaded.

Embedding test queries (original BGE)...
Done. 76 queries.

Embedding test queries (fine-tuned BGE)...
Done. 76 queries.

Re-embedding corpus with fine-tuned BGE...


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Shape: (1476, 768)

Building Mahalanobis graph (original BGE)...
Building mahalanobis graph | N=1476 | K=8
  LedoitWolf...
Edges: 10853

Building Mahalanobis graph (fine-tuned BGE)...
Building mahalanobis graph | N=1476 | K=8
  LedoitWolf...
Edges: 10979

Building retrievers...
All retrievers ready.


Cell 7 — Extract Answers For All Four Methods

In [7]:
from transformers import pipeline as hf_pipeline

print("Loading BERT QA...")
qa_pipe = hf_pipeline(
    "question-answering",
    model  = "deepset/bert-base-uncased-squad2",
    device = 0
)
print("Done.")

OUTPUT_DIR = "/kaggle/working/evaluation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def extract_answer(question, passage, qa_pipe):
    try:
        return qa_pipe(
            question       = question,
            context        = passage,
            max_answer_len = 150
        )["answer"]
    except:
        return ""

def get_hit(results, gold_idx, k=5):
    return int(bool(
        set([r[0] for r in results[:k]]) & gold_idx
    ))

# Storage for all methods
all_rows = {
    "BGE Original"        : [],
    "BGE Fine-tuned"      : [],
    "SSH-RAG Maha Orig"   : [],
    "SSH-RAG Maha FT"     : [],
}

print(f"Extracting answers for {len(dm_test_ids)} questions...")
print("4 methods × questions — takes ~20 minutes\n")

for i, question in enumerate(dm_test_ids):
    gt       = gt_lookup.get(question, "")
    gold_idx = dm_test_r.get(question, set())

    q_emb_orig = dm_q_embs_orig[question]
    q_emb_ft   = dm_q_embs_ft[question]

    # BGE Original
    bge_orig_res     = semantic_retrieve(q_emb_orig, duomind_embeddings)
    bge_orig_passage = chunk_texts[bge_orig_res[0][0]]
    bge_orig_ans     = extract_answer(question, bge_orig_passage, qa_pipe)
    bge_orig_hit     = get_hit(bge_orig_res, gold_idx)

    # BGE Fine-tuned
    bge_ft_res       = semantic_retrieve(q_emb_ft, duomind_emb_ft)
    bge_ft_passage   = chunk_texts[bge_ft_res[0][0]]
    bge_ft_ans       = extract_answer(question, bge_ft_passage, qa_pipe)
    bge_ft_hit       = get_hit(bge_ft_res, gold_idx)

    # SSH-RAG Original BGE
    ssh_orig_res     = ret_ssh_orig.retrieve_from_embedding(
        q_emb_orig, question
    )
    ssh_orig_passage = chunk_texts[ssh_orig_res[0][0]]
    ssh_orig_ans     = extract_answer(question, ssh_orig_passage, qa_pipe)
    ssh_orig_hit     = get_hit(ssh_orig_res, gold_idx)

    # SSH-RAG Fine-tuned BGE
    ssh_ft_res       = ret_ssh_ft.retrieve_from_embedding(
        q_emb_ft, question
    )
    ssh_ft_passage   = chunk_texts[ssh_ft_res[0][0]]
    ssh_ft_ans       = extract_answer(question, ssh_ft_passage, qa_pipe)
    ssh_ft_hit       = get_hit(ssh_ft_res, gold_idx)

    # Store all
    base = {
        "question"           : question,
        "ground_truth_answer": gt,
    }

    all_rows["BGE Original"].append({
        **base,
        "predicted_answer" : bge_orig_ans,
        "top1_passage"     : bge_orig_passage,
        "retrieval_hit_at5": bge_orig_hit,
        "method"           : "BGE Original"
    })
    all_rows["BGE Fine-tuned"].append({
        **base,
        "predicted_answer" : bge_ft_ans,
        "top1_passage"     : bge_ft_passage,
        "retrieval_hit_at5": bge_ft_hit,
        "method"           : "BGE Fine-tuned"
    })
    all_rows["SSH-RAG Maha Orig"].append({
        **base,
        "predicted_answer" : ssh_orig_ans,
        "top1_passage"     : ssh_orig_passage,
        "retrieval_hit_at5": ssh_orig_hit,
        "method"           : "SSH-RAG QA-Maha (original BGE)"
    })
    all_rows["SSH-RAG Maha FT"].append({
        **base,
        "predicted_answer" : ssh_ft_ans,
        "top1_passage"     : ssh_ft_passage,
        "retrieval_hit_at5": ssh_ft_hit,
        "method"           : "SSH-RAG QA-Maha (fine-tuned BGE)"
    })

    if (i+1) % 10 == 0:
        print(f"  [{i+1}/{len(dm_test_ids)}] done")

# Save all
dfs = {}
for method, rows in all_rows.items():
    df = pd.DataFrame(rows)
    fname = method.lower().replace(" ", "_").replace("(","").replace(")","")
    df.to_csv(f"{OUTPUT_DIR}/{fname}_answers.csv", index=False)
    dfs[method] = df
    print(f"{method}: hit@5={df['retrieval_hit_at5'].mean():.4f}")

print("\nAll answer files saved.")

Loading BERT QA...


config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: deepset/bert-base-uncased-squad2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/302 [00:00<?, ?B/s]

... (2 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Done.
Extracting answers for 76 questions...
4 methods × questions — takes ~20 minutes



You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  [10/76] done
  [20/76] done
  [30/76] done
  [40/76] done
  [50/76] done
  [60/76] done
  [70/76] done
BGE Original: hit@5=0.5263
BGE Fine-tuned: hit@5=0.6184
SSH-RAG Maha Orig: hit@5=0.5526
SSH-RAG Maha FT: hit@5=0.5789

All answer files saved.


Cell 8 — Human Evaluation Sheet With All Four Methods

In [8]:
human_eval = pd.DataFrame({
    "id"                     : range(1, len(dm_test_ids)+1),
    "question"               : dfs["BGE Original"]["question"].values,
    "ground_truth_answer"    : dfs["BGE Original"]["ground_truth_answer"].values,

    # BGE Original
    "bge_orig_answer"        : dfs["BGE Original"]["predicted_answer"].values,
    "bge_orig_passage"       : dfs["BGE Original"]["top1_passage"].values,
    "bge_orig_hit"           : dfs["BGE Original"]["retrieval_hit_at5"].values,

    # BGE Fine-tuned
    "bge_ft_answer"          : dfs["BGE Fine-tuned"]["predicted_answer"].values,
    "bge_ft_passage"         : dfs["BGE Fine-tuned"]["top1_passage"].values,
    "bge_ft_hit"             : dfs["BGE Fine-tuned"]["retrieval_hit_at5"].values,

    # SSH-RAG Original
    "ssh_orig_answer"        : dfs["SSH-RAG Maha Orig"]["predicted_answer"].values,
    "ssh_orig_passage"       : dfs["SSH-RAG Maha Orig"]["top1_passage"].values,
    "ssh_orig_hit"           : dfs["SSH-RAG Maha Orig"]["retrieval_hit_at5"].values,

    # SSH-RAG Fine-tuned
    "ssh_ft_answer"          : dfs["SSH-RAG Maha FT"]["predicted_answer"].values,
    "ssh_ft_passage"         : dfs["SSH-RAG Maha FT"]["top1_passage"].values,
    "ssh_ft_hit"             : dfs["SSH-RAG Maha FT"]["retrieval_hit_at5"].values,

    # Annotation columns
    "annotator1_bge_orig"    : "",  # 2=Correct 1=Partial 0=Wrong
    "annotator1_bge_ft"      : "",
    "annotator1_ssh_orig"    : "",
    "annotator1_ssh_ft"      : "",
    "annotator1_notes"       : "",

    "annotator2_bge_orig"    : "",
    "annotator2_bge_ft"      : "",
    "annotator2_ssh_orig"    : "",
    "annotator2_ssh_ft"      : "",
    "annotator2_notes"       : "",
})

human_eval.to_csv(
    f"{OUTPUT_DIR}/human_evaluation_template.csv", index=False
)

print(f"Human eval sheet saved.")
print(f"Rows: {len(human_eval)}")
print(f"Methods per row: 4")
print(f"\nSummary:")
for method, df in dfs.items():
    print(f"  {method}: avg hit@5={df['retrieval_hit_at5'].mean():.4f}")

Human eval sheet saved.
Rows: 76
Methods per row: 4

Summary:
  BGE Original: avg hit@5=0.5263
  BGE Fine-tuned: avg hit@5=0.6184
  SSH-RAG Maha Orig: avg hit@5=0.5526
  SSH-RAG Maha FT: avg hit@5=0.5789


In [9]:
# Re-extract answers for ALL 378 questions
# Not just test split

all_378_ids = list(gold_labels.keys())  # all 378

rows_all = {
    "BGE Original"     : [],
    "BGE Fine-tuned"   : [],
    "SSH-RAG Maha Orig": [],
    "SSH-RAG Maha FT"  : [],
}

print(f"Extracting answers for ALL {len(all_378_ids)} questions...")

for i, question in enumerate(all_378_ids):
    gt       = gt_lookup.get(question, "")
    gold_idx = gold_labels.get(question, set())

    q_emb_orig = dm_q_embs_orig.get(
        question,
        embed_query(question, model, tokenizer)
    )
    q_emb_ft = dm_q_embs_ft.get(
        question,
        ft_bge.encode(
            [f"Represent this sentence for searching "
             f"relevant passages: {question}"],
            normalize_embeddings=True,
            convert_to_numpy=True
        )[0]
    )

    # BGE Original
    bge_orig_res     = semantic_retrieve(q_emb_orig, duomind_embeddings)
    bge_orig_passage = chunk_texts[bge_orig_res[0][0]]
    bge_orig_ans     = extract_answer(question, bge_orig_passage, qa_pipe)

    # BGE Fine-tuned
    bge_ft_res       = semantic_retrieve(q_emb_ft, duomind_emb_ft)
    bge_ft_passage   = chunk_texts[bge_ft_res[0][0]]
    bge_ft_ans       = extract_answer(question, bge_ft_passage, qa_pipe)

    # SSH-RAG Original
    ssh_orig_res     = ret_ssh_orig.retrieve_from_embedding(
        q_emb_orig, question
    )
    ssh_orig_passage = chunk_texts[ssh_orig_res[0][0]]
    ssh_orig_ans     = extract_answer(question, ssh_orig_passage, qa_pipe)

    # SSH-RAG Fine-tuned
    ssh_ft_res       = ret_ssh_ft.retrieve_from_embedding(
        q_emb_ft, question
    )
    ssh_ft_passage   = chunk_texts[ssh_ft_res[0][0]]
    ssh_ft_ans       = extract_answer(question, ssh_ft_passage, qa_pipe)

    base = {
        "question"           : question,
        "ground_truth_answer": gt,
        "is_test_split"      : int(question in dm_test_ids)
    }

    rows_all["BGE Original"].append({
        **base,
        "predicted_answer": bge_orig_ans,
        "top1_passage"    : bge_orig_passage,
        "method"          : "BGE Original"
    })
    rows_all["BGE Fine-tuned"].append({
        **base,
        "predicted_answer": bge_ft_ans,
        "top1_passage"    : bge_ft_passage,
        "method"          : "BGE Fine-tuned"
    })
    rows_all["SSH-RAG Maha Orig"].append({
        **base,
        "predicted_answer": ssh_orig_ans,
        "top1_passage"    : ssh_orig_passage,
        "method"          : "SSH-RAG Maha (original BGE)"
    })
    rows_all["SSH-RAG Maha FT"].append({
        **base,
        "predicted_answer": ssh_ft_ans,
        "top1_passage"    : ssh_ft_passage,
        "method"          : "SSH-RAG Maha (fine-tuned BGE)"
    })

    if (i+1) % 50 == 0:
        print(f"  [{i+1}/{len(all_378_ids)}] done")

# Save
dfs_all = {}
for method, rows in rows_all.items():
    df    = pd.DataFrame(rows)
    fname = method.lower().replace(" ","_").replace("(","").replace(")","")
    df.to_csv(f"{OUTPUT_DIR}/{fname}_all_answers.csv", index=False)
    dfs_all[method] = df
    print(f"Saved: {method} — {len(df)} rows")

# Human eval sheet — all 378
human_eval_full = pd.DataFrame({
    "id"                     : range(1, len(all_378_ids)+1),
    "question"               : dfs_all["BGE Original"]["question"].values,
    "ground_truth_answer"    : dfs_all["BGE Original"]["ground_truth_answer"].values,
    "is_test_split"          : dfs_all["BGE Original"]["is_test_split"].values,
    "bge_orig_answer"        : dfs_all["BGE Original"]["predicted_answer"].values,
    "bge_ft_answer"          : dfs_all["BGE Fine-tuned"]["predicted_answer"].values,
    "ssh_orig_answer"        : dfs_all["SSH-RAG Maha Orig"]["predicted_answer"].values,
    "ssh_ft_answer"          : dfs_all["SSH-RAG Maha FT"]["predicted_answer"].values,
    "bge_orig_passage"       : dfs_all["BGE Original"]["top1_passage"].values,
    "bge_ft_passage"         : dfs_all["BGE Fine-tuned"]["top1_passage"].values,
    "ssh_orig_passage"       : dfs_all["SSH-RAG Maha Orig"]["top1_passage"].values,
    "ssh_ft_passage"         : dfs_all["SSH-RAG Maha FT"]["top1_passage"].values,
    "annotator1_bge_orig"    : "",
    "annotator1_bge_ft"      : "",
    "annotator1_ssh_orig"    : "",
    "annotator1_ssh_ft"      : "",
    "annotator1_notes"       : "",
    "annotator2_bge_orig"    : "",
    "annotator2_bge_ft"      : "",
    "annotator2_ssh_orig"    : "",
    "annotator2_ssh_ft"      : "",
    "annotator2_notes"       : "",
})

human_eval_full.to_csv(
    f"{OUTPUT_DIR}/human_evaluation_full.csv", index=False
)
print(f"\nFull human eval sheet: {len(human_eval_full)} questions")
print(f"Test split questions : {human_eval_full['is_test_split'].sum()}")
print(f"Dev split questions  : {(human_eval_full['is_test_split']==0).sum()}")

Extracting answers for ALL 378 questions...
  [50/378] done
  [100/378] done
  [150/378] done
  [200/378] done
  [250/378] done
  [300/378] done
  [350/378] done
Saved: BGE Original — 378 rows
Saved: BGE Fine-tuned — 378 rows
Saved: SSH-RAG Maha Orig — 378 rows
Saved: SSH-RAG Maha FT — 378 rows

Full human eval sheet: 378 questions
Test split questions : 76
Dev split questions  : 302


In [10]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load E5-large
print("Loading E5-large-v2...")
model_e5 = SentenceTransformer(
    "intfloat/e5-large-v2",
    device="cuda"
)
print("Done.")

# Embed corpus with passage prefix
print("Embedding corpus with E5...")
corpus_emb_e5 = model_e5.encode(
    [f"passage: {t}" for t in chunk_texts],
    batch_size           = 32,
    show_progress_bar    = True,
    normalize_embeddings = True,
    convert_to_numpy     = True
)
print(f"Shape: {corpus_emb_e5.shape}")

# Embed test queries with query prefix
print("Embedding test queries with E5...")
q_embs_e5 = {}
for q in dm_test_ids:
    q_embs_e5[q] = model_e5.encode(
        [f"query: {q}"],
        normalize_embeddings = True,
        convert_to_numpy     = True
    )[0]
print(f"Done. {len(q_embs_e5)} queries.")

# E5 flat baseline
e5_flat_ret = {
    q: semantic_retrieve(q_embs_e5[q], corpus_emb_e5)
    for q in dm_test_ids
}
r_e5_flat = evaluate(
    e5_flat_ret, dm_test_r, dm_test_ids,
    label="E5-large flat baseline"
)

# Build Mahalanobis graph on E5 embeddings
print("\nBuilding E5 Mahalanobis graph...")
graph_e5_maha = LegalDocumentGraph(corpus_emb_e5, "mahalanobis")

# SSH-RAG on E5
ret_e5 = SSHRetriever(
    corpus_texts      = chunk_texts,
    corpus_embeddings = corpus_emb_e5,
    graph             = graph_e5_maha,
    query_model       = model_e5,
    query_tokenizer   = AutoTokenizer.from_pretrained(
        "intfloat/e5-large-v2"
    ),
    decay_lambda      = 0.7,
    alpha             = 0.5
)

e5_ssh_ret = {
    q: ret_e5.retrieve_from_embedding(q_embs_e5[q], q)
    for q in dm_test_ids
}
r_e5_ssh = evaluate(
    e5_ssh_ret, dm_test_r, dm_test_ids,
    label="SSH-RAG QA-Maha (E5-large)"
)

# Summary
print("\n" + "="*55)
print("INDREGBENCH — UPDATED WITH E5 BASELINE")
print("="*55)
print(f"{'Method':<35} {'MAP':>8} {'R@10':>8}")
print("-"*55)
for label, res in [
    ("BGE-base flat",         {"MAP":0.3678,"Recall@10":0.6382}),
    ("E5-large flat",         r_e5_flat),
    ("SSH-RAG Maha (BGE)",    {"MAP":0.3841,"Recall@10":0.5921}),
    ("SSH-RAG Maha (E5)",     r_e5_ssh),
    ("BGE fine-tuned flat",   {"MAP":0.4508,"Recall@10":0.7368}),
]:
    print(f"{label:<35} "
          f"{res['MAP']:>8.4f} "
          f"{res['Recall@10']:>8.4f}")
print("="*55)

Loading E5-large-v2...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

... (6 outputs left out)

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Done.
Embedding corpus with E5...


Batches:   0%|          | 0/47 [00:00<?, ?it/s]

Shape: (1476, 1024)
Embedding test queries with E5...
Done. 76 queries.

=== E5-large flat baseline ===
  MAP         : 0.2748
  Recall@5    : 0.3289
  Recall@10   : 0.4145

Building E5 Mahalanobis graph...
Building mahalanobis graph | N=1476 | K=8
  LedoitWolf...
Edges: 10873

=== SSH-RAG QA-Maha (E5-large) ===
  MAP         : 0.2574
  Recall@5    : 0.3684
  Recall@10   : 0.4474

INDREGBENCH — UPDATED WITH E5 BASELINE
Method                                   MAP     R@10
-------------------------------------------------------
BGE-base flat                         0.3678   0.6382
E5-large flat                         0.2748   0.4145
SSH-RAG Maha (BGE)                    0.3841   0.5921
SSH-RAG Maha (E5)                     0.2574   0.4474
BGE fine-tuned flat                   0.4508   0.7368


In [11]:
from sklearn.utils import resample

def bootstrap_map(retrieved_dict, relevance, 
                  query_ids, n_bootstrap=1000):
    maps = []
    for _ in range(n_bootstrap):
        sample = resample(query_ids, random_state=None)
        r = evaluate(retrieved_dict, relevance, sample)
        maps.append(r["MAP"])
    mean = np.mean(maps)
    std  = np.std(maps)
    ci   = np.percentile(maps, [2.5, 97.5])
    return mean, std, ci

# Run for BGE and SSH-RAG on IndRegBench
print("Bootstrap confidence intervals...")

# BGE Original
bge_ret = {
    q: semantic_retrieve(dm_q_embs_orig[q], duomind_embeddings)
    for q in dm_test_ids
}
bge_mean, bge_std, bge_ci = bootstrap_map(
    bge_ret, dm_test_r, dm_test_ids
)
print(f"BGE Original    : MAP={bge_mean:.4f} ± {bge_std:.4f} "
      f"95%CI=[{bge_ci[0]:.4f}, {bge_ci[1]:.4f}]")

# SSH-RAG QA-Maha
ssh_ret = {
    q: ret_ssh_orig.retrieve_from_embedding(
        dm_q_embs_orig[q], q
    )
    for q in dm_test_ids
}
ssh_mean, ssh_std, ssh_ci = bootstrap_map(
    ssh_ret, dm_test_r, dm_test_ids
)
print(f"SSH-RAG QA-Maha : MAP={ssh_mean:.4f} ± {ssh_std:.4f} "
      f"95%CI=[{ssh_ci[0]:.4f}, {ssh_ci[1]:.4f}]")

# Check if CIs overlap
overlap = ssh_ci[0] < bge_ci[1] and bge_ci[0] < ssh_ci[1]
print(f"\nCIs overlap: {overlap}")
if not overlap:
    print("Improvement is statistically significant.")
else:
    print("CIs overlap — improvement not statistically significant.")
    print("Report honestly in paper.")

Bootstrap confidence intervals...
BGE Original    : MAP=0.3435 ± 0.0459 95%CI=[0.2580, 0.4369]
SSH-RAG QA-Maha : MAP=0.3519 ± 0.0438 95%CI=[0.2661, 0.4480]

CIs overlap: True
CIs overlap — improvement not statistically significant.
Report honestly in paper.


In [12]:
print("Loading BGE-large...")
model_large = SentenceTransformer(
    "BAAI/bge-large-en-v1.5",
    device="cuda"
)
tokenizer_large = AutoTokenizer.from_pretrained(
    "BAAI/bge-large-en-v1.5"
)
print("Done.")

# Embed corpus
print("Embedding corpus with BGE-large...")
corpus_emb_large = model_large.encode(
    chunk_texts,
    batch_size           = 32,
    show_progress_bar    = True,
    normalize_embeddings = True,
    convert_to_numpy     = True
)
print(f"Shape: {corpus_emb_large.shape}")

# Embed test queries
print("Embedding test queries...")
q_embs_large = {}
for q in dm_test_ids:
    q_embs_large[q] = embed_query(q, model_large, tokenizer_large)
print(f"Done. {len(q_embs_large)} queries.")

# Flat BGE-large baseline
bge_large_ret = {
    q: semantic_retrieve(q_embs_large[q], corpus_emb_large)
    for q in dm_test_ids
}
r_bge_large = evaluate(
    bge_large_ret, dm_test_r, dm_test_ids,
    label="BGE-large flat baseline"
)

# Build Mahalanobis graph on BGE-large embeddings
print("\nBuilding Mahalanobis graph on BGE-large...")
graph_large_maha = LegalDocumentGraph(
    corpus_emb_large, "mahalanobis"
)

# SSH-RAG on BGE-large
ret_large = SSHRetriever(
    corpus_texts      = chunk_texts,
    corpus_embeddings = corpus_emb_large,
    graph             = graph_large_maha,
    query_model       = model_large,
    query_tokenizer   = tokenizer_large,
    decay_lambda      = 0.7,
    alpha             = 0.5
)

ssh_large_ret = {
    q: ret_large.retrieve_from_embedding(q_embs_large[q], q)
    for q in dm_test_ids
}
r_ssh_large = evaluate(
    ssh_large_ret, dm_test_r, dm_test_ids,
    label="SSH-RAG QA-Maha (BGE-large)"
)

# Summary
print("\n" + "="*60)
print("INDREGBENCH — BGE-base vs BGE-large comparison")
print("="*60)
print(f"{'Method':<38} {'MAP':>8} {'R@10':>8}")
print("-"*60)
for label, res in [
    ("BM25",
     {"MAP":0.2371,"Recall@10":0.3882}),
    ("BGE-base flat",
     {"MAP":0.3678,"Recall@10":0.6382}),
    ("BGE-large flat",       r_bge_large),
    ("SSH-RAG QA-Maha (BGE-base)",
     {"MAP":0.3841,"Recall@10":0.5921}),
    ("SSH-RAG QA-Maha (BGE-large)", r_ssh_large),
    ("BGE-base fine-tuned",
     {"MAP":0.4508,"Recall@10":0.7368}),
]:
    print(f"{label:<38} "
          f"{res['MAP']:>8.4f} "
          f"{res['Recall@10']:>8.4f}")
print("="*60)

Loading BGE-large...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

... (7 outputs left out)

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Done.
Embedding corpus with BGE-large...


Batches:   0%|          | 0/47 [00:00<?, ?it/s]

Shape: (1476, 1024)
Embedding test queries...
Done. 76 queries.

=== BGE-large flat baseline ===
  MAP         : 0.3348
  Recall@5    : 0.4276
  Recall@10   : 0.5

Building Mahalanobis graph on BGE-large...
Building mahalanobis graph | N=1476 | K=8
  LedoitWolf...
Edges: 11031

=== SSH-RAG QA-Maha (BGE-large) ===
  MAP         : 0.3313
  Recall@5    : 0.4342
  Recall@10   : 0.5066

INDREGBENCH — BGE-base vs BGE-large comparison
Method                                      MAP     R@10
------------------------------------------------------------
BM25                                     0.2371   0.3882
BGE-base flat                            0.3678   0.6382
BGE-large flat                           0.3348   0.5000
SSH-RAG QA-Maha (BGE-base)               0.3841   0.5921
SSH-RAG QA-Maha (BGE-large)              0.3313   0.5066
BGE-base fine-tuned                      0.4508   0.7368


In [13]:
# Quick check
from sentence_transformers import SentenceTransformer
try:
    m = SentenceTransformer("law-ai/InLegalBERT-v2", device="cuda")
    print("InLegalBERT-v2 exists")
except:
    print("Does not exist")

Does not exist


In [14]:
# Separate single-hop vs multi-hop evaluation
# Already have is_cross flag in gold_labels

single_ids = [q for q in dm_test_ids 
              if len(dm_test_r[q]) == 1]
cross_ids  = [q for q in dm_test_ids 
              if len(dm_test_r[q]) == 2]

print(f"Single-hop test: {len(single_ids)}")
print(f"Multi-hop test : {len(cross_ids)}")

# BGE on single-hop
r_bge_single = evaluate(
    {q: semantic_retrieve(dm_q_embs_orig[q], duomind_embeddings)
     for q in single_ids},
    dm_test_r, single_ids,
    label="BGE single-hop"
)

# SSH-RAG on single-hop
r_ssh_single = evaluate(
    {q: ret_ssh_orig.retrieve_from_embedding(
        dm_q_embs_orig[q], q) for q in single_ids},
    dm_test_r, single_ids,
    label="SSH-RAG single-hop"
)

# BGE on multi-hop
r_bge_cross = evaluate(
    {q: semantic_retrieve(dm_q_embs_orig[q], duomind_embeddings)
     for q in cross_ids},
    dm_test_r, cross_ids,
    label="BGE multi-hop"
)

# SSH-RAG on multi-hop
r_ssh_cross = evaluate(
    {q: ret_ssh_orig.retrieve_from_embedding(
        dm_q_embs_orig[q], q) for q in cross_ids},
    dm_test_r, cross_ids,
    label="SSH-RAG multi-hop"
)

print("\n" + "="*55)
print("SINGLE-HOP vs MULTI-HOP ANALYSIS")
print("="*55)
print(f"{'Method':<30} {'MAP':>8} {'R@10':>8}")
print("-"*55)
print(f"{'BGE single-hop':<30} {r_bge_single['MAP']:>8.4f} {r_bge_single['Recall@10']:>8.4f}")
print(f"{'SSH-RAG single-hop':<30} {r_ssh_single['MAP']:>8.4f} {r_ssh_single['Recall@10']:>8.4f}")
print(f"{'BGE multi-hop':<30} {r_bge_cross['MAP']:>8.4f} {r_bge_cross['Recall@10']:>8.4f}")
print(f"{'SSH-RAG multi-hop':<30} {r_ssh_cross['MAP']:>8.4f} {r_ssh_cross['Recall@10']:>8.4f}")
print("="*55)

Single-hop test: 53
Multi-hop test : 23

=== BGE single-hop ===
  MAP         : 0.3516
  Recall@5    : 0.434
  Recall@10   : 0.5283

=== SSH-RAG single-hop ===
  MAP         : 0.3579
  Recall@5    : 0.4717
  Recall@10   : 0.5283

=== BGE multi-hop ===
  MAP         : 0.3304
  Recall@5    : 0.5
  Recall@10   : 0.6087

=== SSH-RAG multi-hop ===
  MAP         : 0.337
  Recall@5    : 0.4348
  Recall@10   : 0.5652

SINGLE-HOP vs MULTI-HOP ANALYSIS
Method                              MAP     R@10
-------------------------------------------------------
BGE single-hop                   0.3516   0.5283
SSH-RAG single-hop               0.3579   0.5283
BGE multi-hop                    0.3304   0.6087
SSH-RAG multi-hop                0.3370   0.5652


In [15]:
from rank_bm25 import BM25Okapi

bm25_dm = BM25Okapi([t.lower().split() for t in chunk_texts])

def knn_retrieve(q_emb, query_text, corpus_emb,
                 bm25, top_k=10):
    # Anchors
    bm25_scores  = bm25.get_scores(query_text.lower().split())
    bm25_anchors = set(np.argsort(bm25_scores)[::-1][:20])
    sem_anchors  = set(np.argsort(corpus_emb @ q_emb)[::-1][:20])
    all_anchors  = bm25_anchors | sem_anchors

    # Flat cosine kNN expansion — no graph, no decay
    expanded = set(all_anchors)
    for anchor in all_anchors:
        neighbors = np.argsort(
            corpus_emb @ corpus_emb[anchor]
        )[::-1][1:6]
        expanded.update(neighbors.tolist())

    # Rerank
    cand_list   = list(expanded)
    final_scores= corpus_emb[cand_list] @ q_emb
    top_idx     = np.argsort(final_scores)[::-1][:top_k]
    return [(cand_list[i], float(final_scores[i]))
            for i in top_idx]

# Evaluate
knn_ret = {
    q: knn_retrieve(
        dm_q_embs_orig[q], q,
        duomind_embeddings, bm25_dm
    )
    for q in dm_test_ids
}
r_knn = evaluate(
    knn_ret, dm_test_r, dm_test_ids,
    label="Simple kNN Expansion (no Mahalanobis)"
)

print("\nComparison:")
print(f"BGE flat           : MAP={0.3678:.4f}")
print(f"Simple kNN expand  : MAP={r_knn['MAP']:.4f}")
print(f"SSH-RAG Maha       : MAP={0.3733:.4f}")
print(f"SSH-RAG QA-Maha    : MAP={0.3841:.4f}")


=== Simple kNN Expansion (no Mahalanobis) ===
  MAP         : 0.3452
  Recall@5    : 0.4539
  Recall@10   : 0.5526

Comparison:
BGE flat           : MAP=0.3678
Simple kNN expand  : MAP=0.3452
SSH-RAG Maha       : MAP=0.3733
SSH-RAG QA-Maha    : MAP=0.3841


In [16]:
recovered    = 0
total_missed = 0

for q in dm_test_ids:
    gold  = dm_test_r.get(q, set())
    if not gold: continue

    q_emb     = dm_q_embs_orig[q]
    bge_top10 = set([r[0] for r in semantic_retrieve(
        q_emb, duomind_embeddings)])
    ssh_top10 = set([r[0] for r in
        ret_ssh_orig.retrieve_from_embedding(q_emb, q)])

    missed    = gold - bge_top10
    recovered += len(missed & ssh_top10)
    total_missed += len(missed)

rate = recovered/total_missed if total_missed > 0 else 0
print(f"Missed by BGE  : {total_missed}")
print(f"Recovered      : {recovered}")
print(f"Recovery rate  : {rate:.1%}")

Missed by BGE  : 43
Recovered      : 2
Recovery rate  : 4.7%


In [19]:
# Add question type to your CSV if available
# If not — use keyword detection

def classify_question(q):
    q_lower = q.lower()
    if any(w in q_lower for w in
           ["what is", "define", "definition", "means"]):
        return "Definition"
    elif any(w in q_lower for w in
             ["must", "shall", "required", "obligation", "duty"]):
        return "Obligation"
    elif any(w in q_lower for w in
             ["how", "procedure", "process", "steps"]):
        return "Procedure"
    elif any(w in q_lower for w in
             ["penalty", "fine", "punish", "consequence"]):
        return "Penalty"
    elif any(w in q_lower for w in
             ["who", "whom", "apply", "scope", "cover"]):
        return "Scope"
    else:
        return "Other"

# Apply to test questions
test_types = {q: classify_question(q) for q in dm_test_ids}

# Evaluate per type
from collections import defaultdict
type_groups = defaultdict(list)
for q, t in test_types.items():
    type_groups[t].append(q)

print("Question type distribution:")
for t, qs in sorted(type_groups.items()):
    print(f"  {t}: {len(qs)}")

print("\n=== PER-TYPE RESULTS ===")
for qtype, qids in sorted(type_groups.items()):
    if len(qids) < 3:
        continue

    bge_ret = {
        q: semantic_retrieve(dm_q_embs_orig[q], duomind_embeddings)
        for q in qids
    }
    ssh_ret = {
        q: ret_ssh_orig.retrieve_from_embedding(
            dm_q_embs_orig[q], q)
        for q in qids
    }

    r_bge = evaluate(bge_ret, dm_test_r, qids)
    r_ssh = evaluate(ssh_ret, dm_test_r, qids)

    print(f"\n{qtype} (n={len(qids)}):")
    print(f"  BGE MAP: {r_bge['MAP']:.4f}")
    print(f"  SSH MAP: {r_ssh['MAP']:.4f}")
    print(f"  Delta  : {r_ssh['MAP']-r_bge['MAP']:+.4f}")

Question type distribution:
  Definition: 11
  Obligation: 20
  Other: 22
  Procedure: 21
  Scope: 2

=== PER-TYPE RESULTS ===

Definition (n=11):
  BGE MAP: 0.2515
  SSH MAP: 0.2614
  Delta  : +0.0099

Obligation (n=20):
  BGE MAP: 0.3479
  SSH MAP: 0.3679
  Delta  : +0.0200

Other (n=22):
  BGE MAP: 0.4266
  SSH MAP: 0.4460
  Delta  : +0.0194

Procedure (n=21):
  BGE MAP: 0.2915
  SSH MAP: 0.2702
  Delta  : -0.0213


In [20]:
# For each chunk, find other chunks from same document
# Add them as candidates after BGE retrieval

def adjacent_section_retrieve(q_emb, query_text,
                               corpus_emb, chunk_ids,
                               bm25, top_k=10):
    """
    Simple same-document expansion baseline.
    No graph, no Mahalanobis — just add chunks from
    same document as top retrieved chunks.
    """
    # Get BGE top-20 anchors
    scores   = corpus_emb @ q_emb
    bge_top  = np.argsort(scores)[::-1][:20]

    # Expand with same-document chunks
    expanded = set(bge_top.tolist())
    for idx in bge_top[:5]:  # expand from top-5 only
        doc_name = chunk_ids[idx].split("_chunk_")[0]
        # Find all chunks from same document
        for j, cid in enumerate(chunk_ids):
            if cid.startswith(doc_name):
                expanded.add(j)

    # Rerank expanded pool
    cand_list    = list(expanded)
    final_scores = corpus_emb[cand_list] @ q_emb
    top_idx      = np.argsort(final_scores)[::-1][:top_k]

    return [(cand_list[i], float(final_scores[i]))
            for i in top_idx]

# Evaluate
adj_ret = {
    q: adjacent_section_retrieve(
        dm_q_embs_orig[q], q,
        duomind_embeddings, chunk_ids, bm25_dm
    )
    for q in dm_test_ids
}
r_adj = evaluate(
    adj_ret, dm_test_r, dm_test_ids,
    label="Adjacent Section Expansion"
)

print(f"\nComparison:")
print(f"BGE flat              : MAP={0.3678:.4f}")
print(f"Adjacent section      : MAP={r_adj['MAP']:.4f}")
print(f"Simple kNN expansion  : MAP={0.3452:.4f}")
print(f"SSH-RAG Mahalanobis   : MAP={0.3733:.4f}")
print(f"SSH-RAG QA-Mahalanobis: MAP={0.3841:.4f}")


=== Adjacent Section Expansion ===
  MAP         : 0.3452
  Recall@5    : 0.4539
  Recall@10   : 0.5526

Comparison:
BGE flat              : MAP=0.3678
Adjacent section      : MAP=0.3452
Simple kNN expansion  : MAP=0.3452
SSH-RAG Mahalanobis   : MAP=0.3733
SSH-RAG QA-Mahalanobis: MAP=0.3841
